In [31]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks
# Obtener la ruta del directorio raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))
# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(PROJECT_ROOT)

INPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'inputs')
OUTPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'outputs')

def get_file_from_path(file_path: str) -> str:
    return os.path.join(INPUTS_FOLDER, file_path)

test_files = [
    'banorte_credit_new.pdf',
    'banorte_credit_old.pdf',
    'banorte_debit.pdf',
    'bbva_debit.pdf',
]

file_path = get_file_from_path('banorte_debit.pdf')

In [32]:
from models import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(file_path)
statement_properties = doc_processor.get_statement_properties()
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']
new_format = statement_properties['new_format']

print(bank, ' - ', statement_type, ' - ', new_format)

banorte  -  debit  -  None


In [33]:
extracted_words = doc_processor.get_extracted_words()

if statement_type == 'debit':
    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_extracted_words.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_extracted_words.csv')
    
extracted_words.to_csv(extracted_words_path, index=False)

In [34]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,1,MARIA,48.816000,51.69000,67.446000,60.6900
1,1,FERNANDA,68.859000,51.69000,98.856000,60.6900
2,1,CANUDAS,100.269000,51.69000,126.882000,60.6900
3,1,ZERTUCHE,128.295000,51.69000,156.294000,60.6900
4,1,CALLE,48.816000,59.40000,65.520000,68.4000
...,...,...,...,...,...,...
1853,6,6/6,557.280000,764.60400,566.739000,773.6040
1854,6,Nuevo,50.400000,772.49955,69.216588,780.4998
1855,6,Leon.,70.688634,772.49955,86.553130,780.4998
1856,6,RFC,88.025176,772.49955,99.017519,780.4998


In [35]:
if statement_type == 'debit':
    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_corrected_words.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_corrected_words.csv')
    
corrected_extracted_words.to_csv(corrected_words_path, index=False)

In [36]:
from models import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructor = table_processor.get_reconstructor()
structured_table = reconstructor.get_structured_table()
structured_table

,Date,Description,MONTO DEL DEPOSITO,MONTO DEL RETIRO,SALDO
0,None,Cuenta Enlace Personal,,,
1,None,FECHA DESCRIPCIÓN / ESTABLECIMIENTO MONTO DEL ...,,,
2,03-FEB-24,SALDO ANTERIOR,,,157.39
3,06-FEB-24,TRASP FONDOS 0000240206 AL R.F.C. CENE7504044...,,,137.39
4,None,Complemento IVA: . 0,,,
5,08-FEB-24,"TRASPASO 0000240208 , DE LA CUENTA: 1256031099",,700.00,837.39
6,08-FEB-24,TRASP FONDOS 0000240208 AL R.F.C. GOGC880322M...,,,57.39
7,None,Tratamientos IVA: . 0,,,
8,13-FEB-24,Deposito en efectivo en cajero Banorte. Te de...,"1,600.00",,"1,657.39"
9,None,MARIA FERNANDA CANUDAS ZERTUCHE en el cajero d...,,,


In [37]:
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,Date,Description,MONTO DEL DEPOSITO,MONTO DEL RETIRO,SALDO
0,03-FEB-24,SALDO ANTERIOR,,,157.39
1,06-FEB-24,TRASP FONDOS 0000240206 AL R.F.C. CENE7504044...,,,137.39
2,08-FEB-24,"TRASPASO 0000240208 , DE LA CUENTA: 1256031099",,700.00,837.39
3,08-FEB-24,TRASP FONDOS 0000240208 AL R.F.C. GOGC880322M...,,,57.39
4,13-FEB-24,Deposito en efectivo en cajero Banorte. Te de...,"1,600.00",,"1,657.39"
5,15-FEB-24,Deposito en efectivo en cajero Banorte. Te de...,"4,050.00",,"5,707.39"
6,15-FEB-24,"TRASPASO 0000240215 , DE LA CUENTA: 1256031099","3,801.50",,"9,508.89"
7,15-FEB-24,"2024021640014TRAP0000466719370 SPEI RECIBIDO,...",,500.00,"10,008.89"
8,15-FEB-24,CARGO POR PAGO CONCENTRACION MOV TARJETA DE C...,,,273.31
9,19-FEB-24,FAR GUAD 2413 RFC:FGU 830930PD3,,,245.31


In [38]:
if statement_type == 'debit':
    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_reconstructed_table.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_reconstructed_table.csv')
    
reconstructed_table.to_csv(reconstructed_table_path, index=False)

In [39]:
from models import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
df_transactions = data_processor.get_normalized_table()
df_transactions

,Date,Description,Amount,Type
0,2024-02-03,SALDO ANTERIOR,157.39,Saldo Inicial
1,2024-02-06,TRASP FONDOS 0000240206 AL R.F.C. CENE7504044...,137.39,Saldo
2,2024-02-08,"TRASPASO 0000240208 , DE LA CUENTA: 1256031099",-700.00,Cargo
3,2024-02-08,TRASP FONDOS 0000240208 AL R.F.C. GOGC880322M...,57.39,Saldo
4,2024-02-13,Deposito en efectivo en cajero Banorte. Te de...,1600.00,Abono
5,2024-02-15,Deposito en efectivo en cajero Banorte. Te de...,4050.00,Abono
6,2024-02-15,"TRASPASO 0000240215 , DE LA CUENTA: 1256031099",3801.50,Abono
7,2024-02-15,"2024021640014TRAP0000466719370 SPEI RECIBIDO,...",-500.00,Cargo
8,2024-02-15,CARGO POR PAGO CONCENTRACION MOV TARJETA DE C...,273.31,Saldo
11,2024-02-19,MERPAGO*MARKET TP RFC:MAG 2105031W3,53.31,Saldo


In [40]:
from models import CsvExporter

exporter = CsvExporter(df_transactions)
if statement_type == 'debit':
    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_normalized_table.csv')
elif statement_type == 'credit':
    _format = 'new' if new_format else 'old'
    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_normalized_table.csv')

exporter.export_data(normalized_table_path)